In [1]:
import os
import h5py
import json
import numpy as np
from PIL import Image
from tqdm import tqdm

from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv

from libero.force.modules import MujocoSensorReader, WrenchObsWrapper

from robokit.debug_utils.printer import print_batch
from robokit.debug_utils.images import (
    plot_action_wrt_time, save_frames_as_video, plot_force_sensor_wrt_time, concatenate_rgb_images
)

[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /mnt/dongxu-fs1/data-ssd/geyuan/programs/anaconda3/envs/libero/lib/python3.8/site-packages/robosuite/scripts/setup_macros.py (__init__.py:9)
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
dataset_root = "/home/geyuan/code/LIBERO-FT/libero/datasets/"
dataset_subname = "libero_90"
hdf5_fn = "KITCHEN_SCENE4_close_the_bottom_drawer_of_the_cabinet_and_open_the_top_drawer_demo.hdf5"
hdf5_path = os.path.join(dataset_root, dataset_subname, hdf5_fn)

with h5py.File(hdf5_path, "r") as f:
    # attributes
    print(list(f["data"].attrs.keys()))
    print(f["data"].attrs["bddl_file_name"])
    print(os.system("pwd"))
    print(os.path.exists(f["data"].attrs["bddl_file_name"]))
    # print(f["data"].attrs["env_args"])
    hdf5_env_meta = json.loads(f["data"].attrs["env_args"])
    hdf5_env_kwargs = hdf5_env_meta["env_kwargs"]
    print(hdf5_env_kwargs)

    # 顶层 keys
    print("Top-level keys:", list(f.keys()))

    # 递归打印整个树结构（group / dataset）
    def print_h5(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"[DSET] {name} shape={obj.shape} dtype={obj.dtype}")
        else:
            print(f"[GRP ] {name}")

    f.visititems(print_h5)


['bddl_file_name', 'env_args', 'env_name', 'macros_image_convention', 'num_demos', 'problem_info', 'tag', 'total']
libero/libero/bddl_files/libero_90/KITCHEN_SCENE4_close_the_bottom_drawer_of_the_cabinet_and_open_the_top_drawer.bddl
/mnt/dongxu-fs1/data-hdd/geyuan/code/LIBERO-FT/debug
0
False
{'robots': ['Panda'], 'controller_configs': {'type': 'OSC_POSE', 'input_max': 1, 'input_min': -1, 'output_max': [0.05, 0.05, 0.05, 0.5, 0.5, 0.5], 'output_min': [-0.05, -0.05, -0.05, -0.5, -0.5, -0.5], 'kp': 150, 'damping_ratio': 1, 'impedance_mode': 'fixed', 'kp_limits': [0, 300], 'damping_ratio_limits': [0, 10], 'position_limits': None, 'orientation_limits': None, 'uncouple_pos_ori': True, 'control_delta': True, 'interpolation': None, 'ramp_ratio': 0.2}, 'bddl_file_name': 'chiliocosm/bddl_files/libero_100/KITCHEN_SCENE4_close_the_bottom_drawer_of_the_cabinet_and_open_the_top_drawer.bddl', 'has_renderer': False, 'has_offscreen_renderer': True, 'ignore_done': True, 'use_camera_obs': True, 'camera_

In [3]:
# build env from task
task_suite = benchmark.get_benchmark_dict()[dataset_subname]()
task_id = 0
task = task_suite.get_task(0)
"""
Task(name='KITCHEN_SCENE4_close_the_bottom_drawer_of_the_cabinet', language='close the bottom drawer of the cabinet', problem='Libero', problem_folder='libero_90', bddl_file='KITCHEN_SCENE4_close_the_bottom_drawer_of_the_cabinet.bddl', init_states_file='KITCHEN_SCENE4_close_the_bottom_drawer_of_the_cabinet.pruned_init')
"""
while task.name not in hdf5_fn.replace(".hdf5", ""):
    task = task_suite.get_task(task_id)
    task_id += 1
print(task)
task_bddl = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)

hdf5_env_kwargs.update({
    "bddl_file_name": task_bddl,
    "camera_heights": 128,
    "camera_widths": 128,
})
if "controller_configs" in hdf5_env_kwargs:
    del hdf5_env_kwargs["controller_configs"]
base_env = OffScreenRenderEnv(**hdf5_env_kwargs)
env = WrenchObsWrapper(base_env, force_sensor="gripper0_force_ee", torque_sensor="gripper0_torque_ee")
env.seed(0)
env.reset()

init_states = task_suite.get_task_init_states(task_id) # for benchmarking purpose, we fix the a set of initial states
init_state_id = 0
env.set_init_state(init_states[init_state_id])

env.env.robots[0].controller.reset_goal()

# vis_images = []
# vis_forces = []
# dummy_action = [0.] * 7
# dummy_action[0] = 0.2  # move forward
# dummy_action[1] = -0.2  # move left
# dummy_action[2] = -0.25  # move down
# for step in tqdm(range(300)):
#     obs, reward, done, info = env.step(dummy_action)
#     if step == 70:
#         dummy_action[0] = -0.2   # moving backward
#         dummy_action[1] = 0.7  # move right
#         dummy_action[2] = 0.   # stop moving down
#     elif step == 130:
#         dummy_action[1] = -0.3  # move left
#         dummy_action[2] = -0.2  # move down
#     resized_image = np.array(
#         Image.fromarray(obs["agentview_image"]).resize((512, 512))
#     )
#     vis_images.append(np.flipud(resized_image))
#     vis_forces.append(obs["wrench_ee"])
#
# force_frames, _, _ = plot_force_sensor_wrt_time(
#     np.array(vis_forces)
# )
# combined_vis_images = [
#     concatenate_rgb_images(img, force_img, resize_ratio=1.)
#     for img, force_img in zip(vis_images, force_frames)
# ]
# save_frames_as_video(combined_vis_images, "tmp_images.mp4", fps=10)

env.close()

Task(name='KITCHEN_SCENE4_close_the_bottom_drawer_of_the_cabinet', language='close the bottom drawer of the cabinet', problem='Libero', problem_folder='libero_90', bddl_file='KITCHEN_SCENE4_close_the_bottom_drawer_of_the_cabinet.bddl', init_states_file='KITCHEN_SCENE4_close_the_bottom_drawer_of_the_cabinet.pruned_init')


Exception ignored in: <function MjRenderContext.__del__ at 0x7f65953570d0>
Traceback (most recent call last):
  File "/mnt/dongxu-fs1/data-ssd/geyuan/programs/anaconda3/envs/libero/lib/python3.8/site-packages/robosuite/utils/binding_utils.py", line 198, in __del__
    self.con.free()
AttributeError: 'MjRenderContextOffscreen' object has no attribute 'con'
/mnt/dongxu-fs1/data-hdd/geyuan/code/LIBERO-FT/libero/libero/benchmark/__init__.py:164: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loade

Replay states from the hdf5 file (Option 1: forward state directly)

In [4]:
base_env = OffScreenRenderEnv(bddl_file_name=task_bddl, camera_heights=128, camera_widths=128)
env = WrenchObsWrapper(base_env, force_sensor="gripper0_force_ee", torque_sensor="gripper0_torque_ee")
env.seed(0)
env.reset()

init_states = task_suite.get_task_init_states(task_id) # for benchmarking purpose, we fix the a set of initial states
init_state_id = 0
env.set_init_state(init_states[init_state_id])

sim = env.env.sim  # robosuite.utils.binding_utils.MjSim
nq, nv = sim.model.nq, sim.model.nv

def interp_states(s0, s1, nq, nv, K):
    # 假设 flatten = [qpos (nq), qvel (nv), ...rest]
    qpos0, qvel0 = s0[:nq], s0[nq:nq+nv]
    qpos1, qvel1 = s1[:nq], s1[nq:nq+nv]
    rest0 = s0[nq+nv:]
    rest1 = s1[nq+nv:]

    # rest 通常可以直接用 s0 的（或者也插值）
    out = []
    for k in range(1, K+1):
        a = k / (K+1)
        qpos = (1-a)*qpos0 + a*qpos1
        qvel = (1-a)*qvel0 + a*qvel1
        rest = (1-a)*rest0 + a*rest1 if rest0.size == rest1.size else rest0
        out.append(np.concatenate([qpos, qvel, rest], axis=0))
    return out


vis_images = []

demo_h5_path = hdf5_path
out_h5_path = "tmp_replayed_wrench.hdf5"
with h5py.File(demo_h5_path, "r") as fin, h5py.File(out_h5_path, "w") as fout:
    # 复制全局 attrs（如果有）
    num_out_attrs = 0
    for k, v in fin.attrs.items():
        fout.attrs[k] = v
        num_out_attrs += 1
    print(f"Copied {num_out_attrs} global attributes from input to output HDF5")

    print("Demo keys:", list(fin["data"].keys()))
    fout.create_group("data")

    for iter_idx, demo_key in enumerate(tqdm(list(fin["data"].keys()))):  # e.g., "demo_0", "demo_1", ...
        print("Processing demo group:", demo_key)
        g_in = fin["data"][demo_key]
        g_out = fout["data"].create_group(demo_key)

        # 复制原有数据集（不破坏结构）
        print("Copying dataset...")
        for k in g_in.keys():  # k in: actions, dones, obs, rewards, robot_states, states
            g_in.copy(k, g_out)
            pass

        # 读取 states
        if "states" not in g_in:
            raise KeyError(f"{demo_key} has no 'states' dataset; cannot do state-replay.")
        states = np.array(g_in["states"], dtype=np.float32)  # (T, state_dim), e.g., (251, 51)
        obs = g_in["obs"]
        print("Original obs keys:", list(obs.keys()))
        print("Original obs['agentview_rgb'] shape:", obs["agentview_rgb"].shape)  # (T,H,W,C), e.g., (217, 128, 128, 3)
        resized_images = [np.flipud(np.array(Image.fromarray(x).resize((512, 512)))) for x in obs["agentview_rgb"]]
        vis_images = resized_images

        # replay -> wrench
        K = 5
        Tlen = states.shape[0]
        wrenches = np.zeros((Tlen, 6), np.float32)
        for t in range(Tlen):
            ## Option 1. Very direct way: set flattened state
            env.set_flattened_state_and_forward(states[t])
            wrenches[t] = env.read_wrench_from_sim()

            ## Option 2. Interp states for smoother sim stepping
            # ws = []
            #
            # env.set_flattened_state_and_forward(states[t])
            # ws.append(env.read_wrench_from_sim())
            #
            # if t < Tlen - 1:
            #     mids = interp_states(states[t], states[t+1], nq, nv, K)
            #     for smid in mids:
            #         env.set_flattened_state_and_forward(smid)
            #         ws.append(env.read_wrench_from_sim())
            #
            # wrenches[t] = np.median(np.stack(ws, axis=0), axis=0)

        # 新增 dataset
        g_out.create_dataset("wrenches", data=wrenches, dtype="float32")

        if iter_idx == 1:
            force_frames, _, _ = plot_force_sensor_wrt_time(wrenches)
            combined_vis_images = [
                concatenate_rgb_images(img, force_img, resize_ratio=1.)
                for img, force_img in zip(vis_images, force_frames)
            ]
            save_frames_as_video(combined_vis_images, f"tmp_replay_images.mp4", fps=10)
            # break  # only do one demo for now

env.close()

KeyboardInterrupt: 

Replay actions from the hdf5 file (Option 2: step with actions)

In [13]:
# base_env = OffScreenRenderEnv(bddl_file_name=task_bddl, camera_heights=128, camera_widths=128)
print(hdf5_env_kwargs)
if "controller_configs" in hdf5_env_kwargs:
    del hdf5_env_kwargs["controller_configs"]
hdf5_env_kwargs["bddl_file_name"] = hdf5_env_kwargs["bddl_file_name"].replace(
    "chiliocosm/bddl_files/libero_100/",
    "/home/geyuan/code/LIBERO-FT/libero/libero/bddl_files/libero_90/"
)
print("[DEBUG] Using bddl file:", hdf5_env_kwargs["bddl_file_name"])
base_env = OffScreenRenderEnv(**hdf5_env_kwargs)
env = WrenchObsWrapper(base_env, force_sensor="gripper0_force_ee", torque_sensor="gripper0_torque_ee")
env.seed(5)
env.reset()

init_states = task_suite.get_task_init_states(task_id) # for benchmarking purpose, we fix the set of initial states
init_state_id = 0
env.set_init_state(init_states[init_state_id])  # len(init_states)=50

sim = env.env.sim  # robosuite.utils.binding_utils.MjSim
nq, nv = sim.model.nq, sim.model.nv

demo_h5_path = hdf5_path
out_h5_path = "tmp_replayed_wrench.hdf5"
with h5py.File(demo_h5_path, "r") as fin, h5py.File(out_h5_path, "w") as fout:
    # 复制全局 attrs（如果有）
    num_out_attrs = 0
    for k, v in fin.attrs.items():
        fout.attrs[k] = v
        num_out_attrs += 1
    print(f"Copied {num_out_attrs} global attributes from input to output HDF5")

    print("Demo keys:", list(fin["data"].keys()))
    fout.create_group("data")

    demo_keys = sorted(list(fin["data"].keys()), key=lambda x: int(x.split("_")[1]))
    for iter_idx, demo_key in enumerate(tqdm(demo_keys)):  # e.g., "demo_0", "demo_1", ...
        vis_hdf5_images = []
        vis_replay_images = []

        print("Processing demo group:", demo_key)
        g_in = fin["data"][demo_key]
        g_out = fout["data"].create_group(demo_key)

        # 复制原有数据集（不破坏结构）
        print("Copying dataset...")
        for k in g_in.keys():  # k in: actions, dones, obs, rewards, robot_states, states
            g_in.copy(k, g_out)
            pass

        # 读取 actions
        if "actions" not in g_in:
            raise KeyError(f"{demo_key} has no 'actions' dataset; cannot do actions-replay.")
        states = np.array(g_in["states"], dtype=np.float64)  # (T, state_dim), e.g., (251, 51)
        actions = np.array(g_in["actions"], dtype=np.float64)  # (T, action_dim), e.g., (251, 7)
        obs = g_in["obs"]
        # print("HDF5 obs keys:", list(obs.keys()))
        # print("HDF5 obs['agentview_rgb'] shape:", obs["agentview_rgb"].shape)  # (T,H,W,C), e.g., (251, 128, 128, 3)
        # print("HDF5 obs['actions'] shape:", actions.shape)
        # print("HDF5 obs['states'] shape:", states.shape)
        resized_images = [np.flipud(np.array(Image.fromarray(x).resize((512, 512)))) for x in obs["agentview_rgb"]]
        vis_hdf5_images = resized_images

        # replay -> wrench
        # In HDF5:
        #    s1 s2 ... s9 s10
        # a0 a1 a2 ... a9
        # In Replay:
        #    s1 s2 ... s9 s10
        #    a1 a2 ... a9
        Tlen = actions.shape[0]
        wrenches = np.zeros((Tlen + 1, 6), np.float32)
        env.reset()
        env.set_flattened_state_and_forward(states[0])  # set as the 1st state of hdf5 demo

        # run 10 zero-action steps to stabilize the sim
        # for _ in range(40):
        #     obs, reward, done, info = env.step(np.zeros_like(actions[0]))
        obs = env.env.regenerate_obs_from_state(states[0])
        # wrenches[0] = obs["wrench_ee"]
        vis_replay_images.append(np.flipud(np.array(Image.fromarray(obs["agentview_image"]).resize((512, 512)))))

        action_shift = 1  # to align with states
        reset_every = 20000  # reset sim state every N steps to reduce compounding errors
        for t in range(action_shift, Tlen):
            if t % reset_every == (reset_every - 1):
                ## Option 1. Very direct way: set flattened state
                env.set_flattened_state_and_forward(states[t - action_shift])
                obs, reward, done, info = env.step(actions[t] * 0.)
                sim_state = env.env.get_sim_state()
                wrenches[t + 1] = env.read_wrench_from_sim()
            else:
                if t % 5 == 4:
                    env.set_flattened_state_and_forward(states[t - action_shift])
                ## Option 2. Step with actions
                obs, reward, done, info = env.step(actions[t])
                sim_state = env.env.get_sim_state()
                wrenches[t + 1] = obs["wrench_ee"]

            vis_replay_images.append(np.flipud(np.array(Image.fromarray(obs["agentview_image"]).resize((512, 512)))))
        success = env.env.check_success()
        print(f"[{demo_key}] success={success}, vis_hdf5 len={len(vis_hdf5_images)}, vis_replay len={len(vis_replay_images)},"
              f" wrench shape={wrenches.shape}")
        # assert len(vis_hdf5_images) == len(vis_replay_images), "Length mismatch between hdf5 images and replay images!"

        # 新增 dataset
        g_out.create_dataset("wrenches", data=wrenches, dtype="float32")

        if iter_idx == 8:
            force_frames, _, _ = plot_force_sensor_wrt_time(wrenches)
            vis_images = [
                concatenate_rgb_images(hdf5_image, replay_image, resize_ratio=1.)
                for hdf5_image, replay_image in zip(vis_hdf5_images, vis_replay_images)
            ]
            combined_vis_images = [
                concatenate_rgb_images(img, force_img, resize_ratio=1.)
                for img, force_img in zip(vis_images, force_frames)
            ]
            save_frames_as_video(combined_vis_images, f"tmp_replay_action_images.mp4", fps=10)
            # break  # only do one demo for now

env.close()

{'robots': ['Panda'], 'bddl_file_name': '/home/geyuan/code/LIBERO-FT/libero/libero/bddl_files/libero_90/KITCHEN_SCENE4_close_the_bottom_drawer_of_the_cabinet_and_open_the_top_drawer.bddl', 'has_renderer': False, 'has_offscreen_renderer': True, 'ignore_done': True, 'use_camera_obs': True, 'camera_depths': False, 'camera_names': ['robot0_eye_in_hand', 'agentview'], 'reward_shaping': True, 'control_freq': 20, 'camera_heights': 128, 'camera_widths': 128, 'camera_segmentations': None}
[DEBUG] Using bddl file: /home/geyuan/code/LIBERO-FT/libero/libero/bddl_files/libero_90/KITCHEN_SCENE4_close_the_bottom_drawer_of_the_cabinet_and_open_the_top_drawer.bddl
Copied 0 global attributes from input to output HDF5
Demo keys: ['demo_0', 'demo_1', 'demo_10', 'demo_11', 'demo_12', 'demo_13', 'demo_14', 'demo_15', 'demo_16', 'demo_17', 'demo_18', 'demo_19', 'demo_2', 'demo_20', 'demo_21', 'demo_22', 'demo_23', 'demo_24', 'demo_25', 'demo_26', 'demo_27', 'demo_28', 'demo_29', 'demo_3', 'demo_30', 'demo_31

  0%|                                                                                                           | 0/50 [00:00<?, ?it/s]

Processing demo group: demo_0
Copying dataset...


  2%|█▉                                                                                                 | 1/50 [00:05<04:50,  5.93s/it]

[demo_0] success=True, vis_hdf5 len=217, vis_replay len=217, wrench shape=(218, 6)
Processing demo group: demo_1
Copying dataset...


...

[demo_47] success=True, vis_hdf5 len=264, vis_replay len=264, wrench shape=(265, 6)
Processing demo group: demo_48
Copying dataset...


 98%|████████████████████████████████████████████████████████████████████████████████████████████████  | 49/50 [04:55<00:05,  5.47s/it]

[demo_48] success=True, vis_hdf5 len=243, vis_replay len=243, wrench shape=(244, 6)
Processing demo group: demo_49
Copying dataset...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [04:59<00:00,  6.00s/it]

[demo_49] success=True, vis_hdf5 len=193, vis_replay len=193, wrench shape=(194, 6)


Check saved wrench data

In [22]:
dataset_root = "/home/geyuan/code/LIBERO-FT/libero/datasets/"
dataset_subname = "libero_90"
hdf5_fn = "KITCHEN_SCENE4_close_the_bottom_drawer_of_the_cabinet_and_open_the_top_drawer_demo.hdf5"
# hdf5_path = os.path.join(dataset_root, dataset_subname, hdf5_fn)
hdf5_path = "tmp_replayed_wrench.hdf5"

def summarize_array(x: np.ndarray):
    """
    x: shape (N, D)
    returns dict of stats per dim
    """
    return {
        "mean": np.mean(x, axis=0),
        "max": np.max(x, axis=0),
        "p01": np.percentile(x, 1, axis=0),
        "p99": np.percentile(x, 99, axis=0),
    }

with h5py.File(hdf5_path, "r") as f:
    # 顶层 keys
    print("Top-level keys:", list(f.keys()))

    # 递归打印整个树结构（group / dataset）
    def print_h5(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"[DSET] {name} shape={obj.shape} dtype={obj.dtype}")
        else:
            print(f"[GRP ] {name}")

    f.visititems(print_h5)

    demos = sorted(f["data"].keys())  # ["demo_0", "demo_1", ...]
    all_w = []

    for dk in demos:
        w = np.array(f["data"][dk]["wrenches"], dtype=np.float32)  # (T,6)
        all_w.append(w)

    W = np.concatenate(all_w, axis=0)  # (sum_T, 6)

    # save images
    demo_keys = sorted(list(f["data"].keys()), key=lambda x: int(x.split("_")[1]))
    for iter_idx, demo_key in enumerate(tqdm(demo_keys, desc="Saving videos")):  # e.g., "demo_0", "demo_1", ...
        obs = f["data"][demo_key]["obs"]
        resized_images = [np.flipud(np.array(Image.fromarray(x).resize((512, 512)))) for x in obs["agentview_rgb"]]
        save_frames_as_video(resized_images, f"saved_replay_action_images_{iter_idx:02d}.mp4", fps=20)

# 分别统计每个维度
stats_w = summarize_array(W)

# 额外统计模长（更有物理意义）
F = W[:, 0:3]
M = W[:, 3:6]
stats_Fnorm = summarize_array(np.linalg.norm(F, axis=1, keepdims=True))  # (N,1)
stats_Mnorm = summarize_array(np.linalg.norm(M, axis=1, keepdims=True))  # (N,1)
stats_Wnorm = summarize_array(np.linalg.norm(W, axis=1, keepdims=True))  # (N,1)

names = ["Fx","Fy","Fz","Mx","My","Mz"]

print("=== Per-dimension wrench stats over ALL demos ===")
for i, n in enumerate(names):
    print(
        f"{n:>2}  mean={stats_w['mean'][i]:9.3f}  "
        f"max={stats_w['max'][i]:9.3f}  "
        f"p01={stats_w['p01'][i]:9.3f}  "
        f"p99={stats_w['p99'][i]:9.3f}"
    )

print("\n=== Norm stats (scalar) ===")
print(f"||F|| mean={stats_Fnorm['mean'][0]:.3f} max={stats_Fnorm['max'][0]:.3f} p01={stats_Fnorm['p01'][0]:.3f} p99={stats_Fnorm['p99'][0]:.3f}")
print(f"||M|| mean={stats_Mnorm['mean'][0]:.3f} max={stats_Mnorm['max'][0]:.3f} p01={stats_Mnorm['p01'][0]:.3f} p99={stats_Mnorm['p99'][0]:.3f}")
print(f"||W|| mean={stats_Wnorm['mean'][0]:.3f} max={stats_Wnorm['max'][0]:.3f} p01={stats_Wnorm['p01'][0]:.3f} p99={stats_Wnorm['p99'][0]:.3f}")

Fn = np.linalg.norm(W[:, :3], axis=1)
print("contact ratio (Fn>1N):", (Fn>1).mean())
print("contact ratio (Fn>10N):", (Fn>10).mean())
print("percentiles:", np.percentile(Fn, [50, 90, 95, 99, 99.5, 99.9]).astype(np.int64))

# per_demo_max = []
# for each demo:
#     per_demo_max.append(np.linalg.norm(w[:, :3], axis=1).max())
# print(np.percentile(per_demo_max, [50, 90, 95, 99]))

Top-level keys: ['data']
[GRP ] data
[GRP ] data/demo_0
[DSET] data/demo_0/actions shape=(217, 7) dtype=float64
[DSET] data/demo_0/dones shape=(217,) dtype=uint8
[GRP ] data/demo_0/obs
[DSET] data/demo_0/obs/agentview_rgb shape=(217, 128, 128, 3) dtype=uint8
[DSET] data/demo_0/obs/ee_ori shape=(217, 3) dtype=float64
[DSET] data/demo_0/obs/ee_pos shape=(217, 3) dtype=float64
[DSET] data/demo_0/obs/ee_states shape=(217, 6) dtype=float64
[DSET] data/demo_0/obs/eye_in_hand_rgb shape=(217, 128, 128, 3) dtype=uint8
[DSET] data/demo_0/obs/gripper_states shape=(217, 2) dtype=float64
[DSET] data/demo_0/obs/joint_states shape=(217, 7) dtype=float64
[DSET] data/demo_0/rewards shape=(217,) dtype=uint8
[DSET] data/demo_0/robot_states shape=(217, 9) dtype=float64
[DSET] data/demo_0/states shape=(217, 51) dtype=float64
[DSET] data/demo_0/wrenches shape=(218, 6) dtype=float32
[GRP ] data/demo_1
[DSET] data/demo_1/actions shape=(192, 7) dtype=float64
[DSET] data/demo_1/dones shape=(192,) dtype=uint8
[G

Saving videos: 100%|███████████████████████████████████████████████████████████████████████████████████| 50/50 [01:30<00:00,  1.82s/it]

=== Per-dimension wrench stats over ALL demos ===
Fx  mean=    1.935  max=  204.044  p01=  -30.874  p99=   48.247
Fy  mean=    3.474  max=  213.431  p01=  -22.003  p99=   55.508
Fz  mean=    5.456  max=  319.855  p01=  -16.546  p99=  102.738
Mx  mean=   -0.247  max=   19.382  p01=   -6.571  p99=    1.933
My  mean=    0.392  max=   43.217  p01=   -7.567  p99=   11.014
Mz  mean=   -0.202  max=   11.862  p01=   -6.148  p99=    4.083

=== Norm stats (scalar) ===
||F|| mean=17.194 max=386.836 p01=4.181 p99=120.624
||M|| mean=1.846 max=45.293 p01=0.079 p99=14.035
||W|| mean=17.323 max=387.508 p01=4.210 p99=121.633
contact ratio (Fn>1N): 1.0
contact ratio (Fn>10N): 0.28613719022244777
percentiles: [  5  48  68 120 146 211]


In [38]:
base_env = OffScreenRenderEnv(bddl_file_name=task_bddl, camera_heights=128, camera_widths=128)
env = WrenchObsWrapper(base_env, force_sensor="gripper0_force_ee", torque_sensor="gripper0_torque_ee")
env.seed(0)
env.reset()

init_states = task_suite.get_task_init_states(task_id) # for benchmarking purpose, we fix the set of initial states
init_state_id = 0
env.set_init_state(init_states[init_state_id])  # len(init_states)=50

sim = env.env.sim  # robosuite.utils.binding_utils.MjSim
nq, nv = sim.model.nq, sim.model.nv

demo_h5_path = hdf5_path
out_h5_path = "tmp_replayed_wrench.hdf5"
with h5py.File(demo_h5_path, "r") as fin, h5py.File(out_h5_path, "w") as fout:
    # 复制全局 attrs（如果有）
    num_out_attrs = 0
    for k, v in fin.attrs.items():
        fout.attrs[k] = v
        num_out_attrs += 1
    print(f"Copied {num_out_attrs} global attributes from input to output HDF5")

    print("Demo keys:", list(fin["data"].keys()))
    fout.create_group("data")

    demo_keys = sorted(list(fin["data"].keys()), key=lambda x: int(x.split("_")[1]))
    for iter_idx, demo_key in enumerate(tqdm(demo_keys)):  # e.g., "demo_0", "demo_1", ...
        vis_hdf5_images = []
        vis_replay_images = []

        print("Processing demo group:", demo_key)
        g_in = fin["data"][demo_key]
        g_out = fout["data"].create_group(demo_key)

        # 复制原有数据集（不破坏结构）
        print("Copying dataset...")
        for k in g_in.keys():  # k in: actions, dones, obs, rewards, robot_states, states
            g_in.copy(k, g_out)
            pass

        # 读取 actions
        if "actions" not in g_in:
            raise KeyError(f"{demo_key} has no 'actions' dataset; cannot do actions-replay.")
        states = np.array(g_in["states"], dtype=np.float32)  # (T, state_dim), e.g., (251, 51)
        actions = np.array(g_in["actions"], dtype=np.float32)  # (T, action_dim), e.g., (251, 7)
        robot_states = np.array(g_in["robot_states"], dtype=np.float32)
        ee_states = np.array(g_in["obs"]["ee_states"], dtype=np.float32)
        ee_pos = np.array(g_in["obs"]["ee_pos"], dtype=np.float32)
        ee_ori = np.array(g_in["obs"]["ee_ori"], dtype=np.float32)
        gripper_states = np.array(g_in["obs"]["gripper_states"], dtype=np.float32)
        print("robot_states:", robot_states[0])
        print("ee_states:", ee_states[0])
        print("ee_pos:", ee_pos[0])
        print("ee_ori:", ee_ori[0])
        print("gripper_states:", gripper_states[0])
        obs = g_in["obs"]
        # print("HDF5 obs keys:", list(obs.keys()))
        # print("HDF5 obs['agentview_rgb'] shape:", obs["agentview_rgb"].shape)  # (T,H,W,C), e.g., (251, 128, 128, 3)
        # print("HDF5 obs['actions'] shape:", actions.shape)
        # print("HDF5 obs['states'] shape:", states.shape)
        resized_images = [np.flipud(np.array(Image.fromarray(x).resize((512, 512)))) for x in obs["agentview_rgb"]]
        vis_hdf5_images = resized_images

        # replay -> wrench
        # In HDF5:
        #    s1 s2 ... s9 s10
        # a0 a1 a2 ... a9
        # In Replay:
        #    s1 s2 ... s9 s10
        #    a1 a2 ... a9
        Tlen = actions.shape[0]
        wrenches = np.zeros((Tlen + 1, 6), np.float32)
        env.reset()
        env.set_flattened_state_and_forward(states[0])  # set as the 1st state of hdf5 demo

        # run 10 zero-action steps to stabilize the sim
        for _ in range(10):
            obs, reward, done, info = env.step(np.zeros_like(actions[0]))
        sim_state = env.env.get_sim_state()
        wrenches[0] = obs["wrench_ee"]
        print("obs.robot0_eef_quat:", obs["robot0_eef_quat"])
        # print_batch("obs", obs)
        # print_batch("reward", reward)
        break
        vis_replay_images.append(np.flipud(np.array(Image.fromarray(obs["agentview_image"]).resize((512, 512)))))

        action_shift = 0  # to align with states
        for t in range(action_shift, Tlen):
            ## Option 2. Step with actions
            obs, reward, done, info = env.step(actions[t])
            wrenches[t + 1] = obs["wrench_ee"]
            vis_replay_images.append(np.flipud(np.array(Image.fromarray(obs["agentview_image"]).resize((512, 512)))))
        success = env.env.check_success()
        print(f"[{demo_key}] success={success}, vis_hdf5 len={len(vis_hdf5_images)}, vis_replay len={len(vis_replay_images)},"
              f" wrench shape={wrenches.shape}")
        # assert len(vis_hdf5_images) == len(vis_replay_images), "Length mismatch between hdf5 images and replay images!"

        # 新增 dataset
        g_out.create_dataset("wrenches", data=wrenches, dtype="float32")

        if iter_idx == 1:
            force_frames, _, _ = plot_force_sensor_wrt_time(wrenches)
            vis_images = [
                concatenate_rgb_images(hdf5_image, replay_image, resize_ratio=1.)
                for hdf5_image, replay_image in zip(vis_hdf5_images, vis_replay_images)
            ]
            combined_vis_images = [
                concatenate_rgb_images(img, force_img, resize_ratio=1.)
                for img, force_img in zip(vis_images, force_frames)
            ]
            save_frames_as_video(combined_vis_images, f"tmp_replay_action_images.mp4", fps=10)
            # break  # only do one demo for now

env.close()

Copied 0 global attributes from input to output HDF5
Demo keys: ['demo_0', 'demo_1', 'demo_10', 'demo_11', 'demo_12', 'demo_13', 'demo_14', 'demo_15', 'demo_16', 'demo_17', 'demo_18', 'demo_19', 'demo_2', 'demo_20', 'demo_21', 'demo_22', 'demo_23', 'demo_24', 'demo_25', 'demo_26', 'demo_27', 'demo_28', 'demo_29', 'demo_3', 'demo_30', 'demo_31', 'demo_32', 'demo_33', 'demo_34', 'demo_35', 'demo_36', 'demo_37', 'demo_38', 'demo_39', 'demo_4', 'demo_40', 'demo_41', 'demo_42', 'demo_43', 'demo_44', 'demo_45', 'demo_46', 'demo_47', 'demo_48', 'demo_49', 'demo_5', 'demo_6', 'demo_7', 'demo_8', 'demo_9']


  0%|                                                                                                           | 0/50 [00:00<?, ?it/s]

Processing demo group: demo_0
Copying dataset...
robot_states: [ 0.03626977 -0.03618784 -0.20186226  0.01013853  1.1801711   0.9996168
 -0.00674171 -0.02033794 -0.0175275 ]
ee_states: [-0.20186226  0.01013853  1.1801711   3.17592    -0.02141934 -0.06461642]
ee_pos: [-0.20186226  0.01013853  1.1801711 ]
ee_ori: [ 3.17592    -0.02141934 -0.06461642]
gripper_states: [ 0.03626977 -0.03618784]


  0%|                                                                                                           | 0/50 [00:01<?, ?it/s]

obs.robot0_eef_quat: [ 9.99617687e-01  2.46561481e-04 -2.76432318e-02 -5.20345748e-04]


Check physical parameters in the sim

In [4]:
from libero.force.physics_helper import PhysicsHelper
base_env = OffScreenRenderEnv(bddl_file_name=task_bddl, camera_heights=128, camera_widths=128)
env = WrenchObsWrapper(base_env, force_sensor="gripper0_force_ee", torque_sensor="gripper0_torque_ee")
env.seed(0)
env.reset()

init_states = task_suite.get_task_init_states(task_id) # for benchmarking purpose, we fix the a set of initial states
init_state_id = 0
env.set_init_state(init_states[init_state_id])

sim = env.env.sim  # robosuite.utils.binding_utils.MjSim
nq, nv = sim.model.nq, sim.model.nv
print(type(sim.model))


''' Find out physical parameters of the env '''
model = env.env.sim.model
phy_helper = PhysicsHelper(
    model,
    object_keywords=("drawer", "cabinet"),
)
phy_helper.apply_dynamics_shift(
)
phy_helper.restore_original_params()

env.close()

<class 'robosuite.utils.binding_utils.MjModel'>
[DEBUG] All joints: 0:robot0_joint1; 1:robot0_joint2; 2:robot0_joint3; 3:robot0_joint4; 4:robot0_joint5; 5:robot0_joint6; 6:robot0_joint7; 7:gripper0_finger_joint1; 8:gripper0_finger_joint2; 9:akita_black_bowl_1_joint0; 10:wine_bottle_1_joint0; 11:white_cabinet_1_top_level; 12:white_cabinet_1_middle_level; 13:white_cabinet_1_bottom_level
[DEBUG] All geoms: 0:floor; 1:; 2:; 3:; 4:; 5:; 6:wall_left_visual; 7:wall_right_visual; 8:wall_rear_visual; 9:wall_front_visual; 10:table_collision; 11:table_visual; 12:table_leg1_visual; 13:table_leg2_visual; 14:table_leg3_visual; 15:table_leg4_visual; 16:robot0_g0_vis; 17:robot0_g1_vis; 18:robot0_g2_vis; 19:robot0_g3_vis; 20:robot0_g4_vis; 21:robot0_g5_vis; 22:robot0_g6_vis; 23:robot0_g7_vis; 24:robot0_g8_vis; 25:robot0_g9_vis; 26:robot0_g10_vis; 27:robot0_g11_vis; 28:robot0_link0_collision; 29:robot0_g12_vis; 30:robot0_link1_collision; 31:robot0_g13_vis; 32:robot0_link2_collision; 33:robot0_g14_vis; 3

In [39]:
''' quant - euler '''
import robosuite.utils.transform_utils as T
def quat_xyzw_to_rotvec(q_xyzw: np.ndarray) -> np.ndarray:
    """
    输入: quat in xyzw
    输出: rotation vector (axis-angle), shape (3,) = theta * axis
    """
    # 有些 robosuite 版本函数名叫 quat2axisangle
    rotvec = T.quat2axisangle(q_xyzw)   # (3,)
    return rotvec

q_xyzw = np.array([0.9996168, -0.00674171, -0.02033794, -0.0175275 ])
euler_xyz = quat_xyzw_to_rotvec(q_xyzw)
print("euler_xyz (radians):", euler_xyz)

obs_q_xyzw = np.array([9.99617687e-01,  2.46561481e-04, -2.76432318e-02, -5.20345748e-04])
obs_euler_xyz = quat_xyzw_to_rotvec(q_xyzw)
print("obs_euler_xyz (radians):", obs_euler_xyz)

euler_xyz (radians): [ 3.17592004 -0.02141934 -0.06461643]
obs_euler_xyz (radians): [ 3.17592004 -0.02141934 -0.06461643]
